In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table


## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 15:27:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [3]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [4]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

## Build Bronze Table

In [5]:
# # create bronze datalake
# bronze_lms_directory = "datamart/bronze/lms/"

# if not os.path.exists(bronze_lms_directory):
#     os.makedirs(bronze_lms_directory)

In [6]:
# run bronze backfill
for date_str in dates_str_lst:
    last_bronze_output = utils.data_processing_bronze_table.process_bronze_table_features(date_str, spark)


2023-01-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_01_01.csv

2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_attributes2023_01_01.csv

2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_financials2023_01_01.csv

2023-02-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_02_01.csv

2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_attributes2023_02_01.csv

2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_financials2023_02_01.csv

2023-03-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_03_01.csv

2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_attributes2023_03_01.csv

2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_financials2023_03_01.csv

2023-04-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_04_01.csv

2023-04-01	row count: 510
saved to: datamart/bronze/bro

In [7]:
for key in last_bronze_output.keys():
    last_bronze_output[key].show(30, truncate=False)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|145 |189 |109 |134 |196 |-37 |101 |82  |111 |24   |-26  |-17  |65   |249  |200  |185  |-83  |-18  |-76  |30   |CUS_0x1037 |2024-12-01   |
|40  |184 |187 |75  |192 |146 |38  |109 |353 |141  |-9   |-22  |-14  |193  |125  |117  |215  |91   |33   |255  |CUS_0x1069 |2024-12-01   |
|98  |121 |180 |200 |95  |48  |59  |194 |76  |84   |298  |-57  |167  |101  |92   |185  |98   |68   |-60  |116  |CUS_0x114a |2024-12-01   |
|85  |96  |19  |47  |30  |39  |-32 |210 |-81 |206  |37   |105  |143  |94   |139  |237  |78   |187  |77   |33   |CUS_0x1184 |2024-12-01   |
|98  |45  |155 |56  |112 |4

In [8]:
# # inspect output
# utils.data_processing_bronze_table.process_bronze_table(date_str, bronze_lms_directory, spark).toPandas()

## Build Silver Table

In [9]:
# # create bronze datalake
# silver_loan_daily_directory = "datamart/silver/loan_daily/"

# if not os.path.exists(silver_loan_daily_directory):
#     os.makedirs(silver_loan_daily_directory)

In [10]:
import importlib
importlib.reload(utils.data_processing_silver_table)

# run silver backfill
for date_str in dates_str_lst:
    utils.data_processing_silver_table.process_silver_table_features(date_str, spark)


loaded from: datamart/bronze/bronze_feature_clickstream2023_01_01.csv row count: 8974


saved to: datamart/silver/silver_feature_clickstream2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_01_01.csv row count: 530
saved to: datamart/silver/silver_features_attributes2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_01_01.csv row count: 530
saved to: datamart/silver/silver_features_financials2023_01_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_02_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_attributes2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_financials2023_02_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_03_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_03_01

In [ ]:
# utils.data_processing_silver_table.process_silver_table(date_str, bronze_lms_directory, silver_loan_daily_directory, spark).toPandas()

## EDA on credit labels

In [ ]:
# set dpd label definition
dpd = 30

# Path to the folder containing CSV files
folder_path = silver_loan_daily_directory

# Read all CSV files into a single DataFrame
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)

# filter only completed loans
df = df.filter(col("loan_start_date") < datetime.strptime("2024-01-01", "%Y-%m-%d"))

# create dpd flag if more than dpd
df = df.withColumn("dpd_flag", F.when(col("dpd") >= dpd, 1).otherwise(0))

# actual bads 
actual_bads_df = df.filter(col("installment_num") == 10)

# prepare for analysis
# df = df.filter(col("installment_num") < 10)

# visualise bad rate
pdf = df.toPandas()

# Group by col_A and count occurrences in col_B
grouped = pdf.groupby('mob')['dpd_flag'].mean()

# Sort the index (x-axis) of the grouped DataFrame
grouped = grouped.sort_index()

# Plotting
grouped.plot(kind='line', marker='o')

plt.title('DPD: '+ str(dpd))
plt.xlabel('mob')
plt.ylabel('bad rate')
plt.grid(True)
plt.show()


In [ ]:
df.show()

## Build gold table for labels

In [ ]:
# create bronze datalake
gold_label_store_directory = "datamart/gold/label_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

In [ ]:
# run gold backfill
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_labels_gold_table(date_str, silver_loan_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6)


In [ ]:
utils.data_processing_gold_table.process_labels_gold_table(date_str, silver_loan_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6).dtypes


## inspect label store

In [ ]:
folder_path = gold_label_store_directory
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

In [ ]:
df.printSchema()